# IB-DTA — Interaction Bottlenecks for Drug–Target Affinity

Run this top to bottom. It installs everything, downloads a real dataset,
trains the model on a subset, and runs the faithfulness audit — about
**10–15 minutes on a free Colab CPU**, faster on a GPU.

**Base paper:** Bai, Miljković, John & Lu (2023), *Interpretable bilinear
attention network with domain adaptation improves drug–target prediction*,
**Nature Machine Intelligence**.
[Paper](https://www.nature.com/articles/s42256-022-00605-1) ·
[Code](https://github.com/peizhenbai/DrugBAN)

**What IB-DTA changes.** DrugBAN builds an atom × residue interaction map,
then contracts it into a joint vector before predicting. After that
contraction no single contact is on the path to the output, so the
attention map *correlates* with the prediction rather than producing it.
IB-DTA removes the contraction: affinity is a sum of signed contributions
over a sparse selected set of pairs. Deleting a contact changes the output
by exactly that contact's contribution.

No `dgl` required — the encoders here are plain PyTorch, which is why this
notebook installs in seconds rather than fighting CUDA wheels.

## 1. Setup

In [ ]:
# Only rdkit is needed beyond Colab's defaults.
!pip install -q rdkit pyyaml 2>/dev/null
import torch, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("python", sys.version.split()[0])

In [ ]:
# Get the IB-DTA source straight from GitHub.
import os

REPO = "https://github.com/BabarAli13/ib-dta.git"
if not os.path.isdir("/content/ib-dta"):
    !git clone -q $REPO /content/ib-dta
%cd /content/ib-dta
!git log --oneline -1

# Confirm the memory fix is present. If this prints MISSING, push the
# updated configs/ib_layer.py first or training will need ~30 GB.
import yaml
cfg = yaml.safe_load(open("configs/IBDTA.yaml"))
print("CHECKPOINT_CHUNKS:",
      cfg["IB"].get("CHECKPOINT_CHUNKS", "MISSING - update the repo"))


## 2. Sanity check

These tests verify the properties every later claim depends on. If any
fails, stop — something is wrong with the environment, not the science.

In [ ]:
!python test_ib.py

## 2b. Reproduce the theory table (no training, no GPU)

These three scripts reproduce every residual in Table 1 of the paper.
They run on CPU in roughly half an hour and need no dataset.


In [ ]:
!python verify_theory.py
!python verify_generalization.py
!python verify_identifiability.py


## 3. Download Davis

Davis: 68 kinase inhibitors × 442 protein entries, Kd converted to pKd.
Source is the DeepDTA release (Öztürk et al. 2018).

In [ ]:
!python download_data.py --datasets davis

### Data integrity check — run this before trusting any metric

Davis lists 442 protein *entries* but fewer unique *sequences*. Isoforms
and mutants with identical sequences collapse onto the same key once you
index by sequence text, which is what a sequence-based model actually
sees. The result is pairs with identical inputs and different labels:
an irreducible error floor, and leakage across a random split.

In [ ]:
import pandas as pd, splits

df = pd.read_csv("datasets/davis/full.csv")
summary, dup = splits.audit_duplicates(df)
for k, v in summary.items():
    print(f"{k:24s} {v:,.4f}" if isinstance(v, float) else f"{k:24s} {v:,}")

print("\nWorst disagreements (same drug, same sequence, different pKd):")
print(dup.nlargest(5, "spread")[["n", "lo", "hi", "spread"]].to_string(index=False))

## 4. Build honest splits

`random` is reported for comparability only. Every claim should rest on
the cold and scaffold regimes. The audit prints exactly how much of each
test set the model has already seen.

In [ ]:
import pandas as pd, splits

df = pd.read_csv("datasets/davis/full.csv")
for regime in ["random", "cold_drug", "cold_target", "scaffold"]:
    tr, va, te, rep = splits.write_splits(df, "datasets/davis", regime, seed=42)
    t = rep[rep.split == "test"].iloc[0]
    print(f"{regime:12s} {len(tr):6,}/{len(va):5,}/{len(te):6,}   "
          f"drug-overlap {t.drug_overlap_pct:5.1f}%   "
          f"target-overlap {t.target_overlap_pct:5.1f}%")

Note the random split: on Davis it leaks essentially everything, because
the matrix is dense and there are only 68 distinct drugs. This is why
random-split Davis numbers in the literature are hard to interpret.

## 5. Train

Small subset and few epochs so the notebook finishes quickly. Raise
`--subset` and `--epochs` (or drop them entirely) on a GPU.

**Watch the `val_support` column.** It is the mean number of interaction
pairs the model actually uses. If it collapses toward 1, the model has
degenerated into an argmax and its explanations are empty, regardless of
what the loss says.

In [ ]:
!python train_ib.py --cfg configs/IBDTA.yaml --data davis --split cold_target \
    --subset 1500 --eval-subset 500 --epochs 8 \
    --set IB.H_DIM 64 PROTEIN.MAX_LEN 400 SOLVER.BATCH_SIZE 16

## 6. The faithfulness audit

This is the part that distinguishes IB-DTA from a model that merely
*displays* attention.

Delete the contacts an explanation ranks highest and watch the prediction
move. Compare against deleting at random. A faithful explanation moves it
faster than chance; an unfaithful one does not.

**AFS** (above-random faithfulness) is the normalised gap between the two
curves. AFS ≤ 0 means the explanation is no better than chance.

In [ ]:
import numpy as np, torch, pandas as pd
import faithfulness as FA
from encoders import DTADataset, collate
from models_ib import IBDTA
from train_ib import load_cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = load_cfg("configs/IBDTA.yaml", {"IB.H_DIM": 64, "PROTEIN.MAX_LEN": 400})

model = IBDTA(cfg).to(device)
model.load_state_dict(torch.load("result/davis/cold_target/best.pt",
                                 map_location=device))
model.eval()

te = pd.read_csv("datasets/davis/cold_target/test.csv").sample(16, random_state=0)
ds = DTADataset(te, max_atoms=cfg["DRUG"]["MAX_NODES"],
                max_protein=cfg["PROTEIN"]["MAX_LEN"], verbose=False)
nf, adj, nm, tok, pl, y = collate([ds[i] for i in range(len(ds))])
nf, adj, nm, tok, pl = [t.to(device) for t in (nf, adj, nm, tok, pl)]

def predict(drop):
    with torch.no_grad():
        out, _, _ = model(nf, adj, nm, tok, pl, drop_pairs=drop)
    return out

with torch.no_grad():
    _, contrib, alpha = model(nf, adj, nm, tok, pl)

res = FA.audit(predict, contrib, seed=0)
print(FA.format_report("IB-DTA", res))
print(f"\nmean support: {model.support_size(alpha).mean():.1f} pairs "
      f"out of {alpha.shape[2] * alpha.shape[3]:,} candidates")

### The exactness guarantee

For an additive bottleneck the change from deleting a set of contacts is
*analytically* known: it equals the sum of their contributions. No
post-hoc attribution method can promise this. A pooled model fails the
same check, and the size of the failure measures how much prediction mass
bypasses the interaction map.

In [ ]:
drop = torch.zeros_like(contrib, dtype=torch.bool)
flat = contrib.reshape(contrib.shape[0], -1)
idx = flat.abs().topk(5, dim=-1).indices
drop.reshape(contrib.shape[0], -1).scatter_(1, idx, True)

max_res, ok, detail = FA.check_additive_exactness(predict, contrib, drop)
print("additive deletion bound:", "EXACT" if ok else "VIOLATED")
for k, v in detail.items():
    print(f"  {k:22s} {v:.3e}")

### Plot the deletion curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.6), dpi=120)
ax.plot(res["fractions"] * 100, res["most"], "o-", label="delete most important")
ax.plot(res["fractions"] * 100, res["random"], "s--", color="gray",
        label="delete at random")
ax.plot(res["fractions"] * 100, res["least"], "^:", color="green",
        label="delete least important")
ax.set_xlabel("% of interaction pairs deleted")
ax.set_ylabel("|change in prediction| (pKd)")
ax.set_title(f"Deletion curves — AFS = {res['afs']:+.3f}")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 7. What the model is looking at

Each selected contact carries a **signed contribution in pKd units**, so
the model states not just *where* it looks but *how much that contact is
worth*. No pooled DTA model can make that statement.

In [ ]:
from ib_layer import TopPairs

vals, idx = TopPairs.topk(contrib, 8)
print(f"{'sample':>7} {'head':>5} {'atom':>6} {'residue':>8} {'contribution (pKd)':>20}")
for b in range(min(3, contrib.shape[0])):
    for r in range(5):
        h, a, q = idx[b, r].tolist()
        print(f"{b:>7} {h:>5} {a:>6} {q:>8} {vals[b, r].item():>+20.4f}")
    print()

In [ ]:
# Where selection mass sits, for one complex
import numpy as np
b = 0
A = alpha[b, 0].detach().cpu().numpy()
nz = A > 0
fig, ax = plt.subplots(figsize=(7, 3.2), dpi=120)
im = ax.imshow(np.where(nz, A, np.nan), aspect="auto", cmap="viridis",
               interpolation="nearest")
ax.set_xlabel("residue position"); ax.set_ylabel("atom index")
ax.set_title(f"Selected contacts — {nz.sum()} of {A.size:,} pairs "
             f"({100 * nz.sum() / A.size:.2f}%)")
plt.colorbar(im, ax=ax, label="selection weight")
plt.tight_layout(); plt.show()

## 8. Full experiment suite (optional)

Reproduces every figure in `figures/`. On a CPU keep the subsets small;
on a GPU use `--full`.

In [ ]:
!python run_experiments.py --only e0 e5 --n-train 300 --n-eval 150 --epochs 4
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("figures/*.png")):
    print(f); display(Image(f))

## Next steps

- `--full` on a GPU for real numbers; everything above is demo scale.
- Run the deletion audit against **published** interpretable DTA models.
  If their explanations score AFS near zero, that is a result on its own,
  independent of whether IB-DTA wins on accuracy.
- Sweep `IB.BUDGET_K` to trace the faithfulness–accuracy frontier.

### Citation of the base model

```bibtex
@article{bai2023drugban,
  title   = {Interpretable bilinear attention network with domain adaptation
             improves drug-target prediction},
  author  = {Bai, Peizhen and Miljkovi{\'c}, Filip and John, Bino and Lu, Haiping},
  journal = {Nature Machine Intelligence},
  volume  = {5}, pages = {126--136}, year = {2023},
  doi     = {10.1038/s42256-022-00605-1}
}
```